# 03 — Exploratory Data Analysis (development period)

M4 task 1. Explore the 10 selected categories using ONLY the development
window (no validation/test leakage), and produce five figures:

- category concentration
- zero-purchase ratio
- weekday effect
- trend
- view -> cart -> purchase funnel

The standalone script `scripts/run_eda.py` runs the same logic and writes
PNGs under `reports/figures/`; this notebook renders the figures inline and
records the conclusions. Requires numpy + matplotlib (same environment as
`scripts/run_eda.py`).


In [ ]:
import csv
import re
from collections import defaultdict
from datetime import date
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
print("project root:", PROJECT_ROOT)

FIELDS = ["date", "category_id", "category_code", "views", "cart", "purchase"]
EVENTS = FIELDS[3:]
WEEKDAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]


In [ ]:
def load_eda_config(path):
    text = Path(path).read_text(encoding="utf-8")
    try:
        import yaml  # optional; fallback below reads the same keys with regex
        return yaml.safe_load(text)
    except ImportError:
        pass

    def first(pattern):
        m = re.search(pattern, text, re.M)
        return m.group(1) if m else None

    # First occurrence in the file belongs to the primary multi_store block
    # (the cosmetics block appears later in config.yaml).
    return {
        "paths": {"processed_dir": "data/processed"},
        "m3": {"sources": {"multi_store": {
            "development_start": first(r"development_start:\s*'([^']+)'"),
            "development_end": first(r"development_end:\s*'([^']+)'"),
            "variants": {"10": {"output": "daily_category.csv"}},
        }}},
    }


cfg = load_eda_config(PROJECT_ROOT / "config" / "config.yaml")
src = cfg["m3"]["sources"]["multi_store"]
dev_start = src["development_start"]
dev_end = src["development_end"]
csv_path = PROJECT_ROOT / cfg["paths"]["processed_dir"] / src["variants"]["10"]["output"]
print("dev window:", dev_start, "..", dev_end)
print("daily table:", csv_path)


In [ ]:
def read_daily(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        missing = set(FIELDS) - set(reader.fieldnames or [])
        if missing:
            raise ValueError("Missing daily columns: " + ",".join(sorted(missing)))
        return list(reader)


def to_numeric(rows):
    return [
        {
            "date": date.fromisoformat(r["date"]),
            "category_id": r["category_id"],
            "category_code": r["category_code"],
            "views": int(r["views"]),
            "cart": int(r["cart"]),
            "purchase": int(r["purchase"]),
        }
        for r in rows
    ]


def filter_dev(rows, dev_start, dev_end):
    a, b = date.fromisoformat(dev_start), date.fromisoformat(dev_end)
    return [r for r in rows if a <= r["date"] <= b]


rows = filter_dev(to_numeric(read_daily(csv_path)), dev_start, dev_end)
n_cats = len({r["category_id"] for r in rows})
n_days = len({r["date"] for r in rows})
print(f"rows={len(rows)} categories={n_cats} days={n_days} window={dev_start}..{dev_end}")


In [ ]:
def category_info(rows):
    """Per-category totals + day/zero-day counts."""
    info = defaultdict(lambda: {"code": "", "views": 0, "cart": 0, "purchase": 0, "days": 0, "zero": 0})
    for r in rows:
        c = info[r["category_id"]]
        c["code"] = r["category_code"]
        c["views"] += r["views"]
        c["cart"] += r["cart"]
        c["purchase"] += r["purchase"]
        c["days"] += 1
        if r["purchase"] == 0:
            c["zero"] += 1
    return info


def short_label(cid, code):
    """Short label: last code token, or a compact id when the code is empty."""
    return code.split(".")[-1] if code else f"id:{cid[-6:]}"


In [ ]:
# --- Plot 1: category concentration ---
info = category_info(rows)
items = sorted(info.items(), key=lambda kv: kv[1]["purchase"], reverse=True)
labels = [short_label(cid, v["code"]) for cid, v in items]
purchases = np.array([v["purchase"] for _, v in items], dtype=float)
total = purchases.sum()
shares = purchases / total * 100.0
cum = np.cumsum(shares)

fig, ax1 = plt.subplots(figsize=(9, 5))
x = np.arange(len(items))
ax1.bar(x, purchases, color="#4C72B0")
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45, ha="right")
ax1.set_ylabel("Total purchase events (dev period)")
ax1.set_title("Category concentration (dev-period purchases)")

ax2 = ax1.twinx()
ax2.plot(x, cum, color="#C44E52", marker="o", label="Cumulative share")
ax2.set_ylabel("Cumulative share %")
ax2.set_ylim(0, 105)
ax2.axhline(100, color="grey", lw=0.6, ls="--")

for xi, (p, s) in enumerate(zip(purchases, shares)):
    if s >= 4.0:
        ax1.annotate(f"{s:.1f}%", (xi, p * 0.55), ha="center", va="center",
                     color="white", fontsize=8, fontweight="bold")
    else:
        ax1.annotate(f"{s:.1f}%", (xi, p), ha="center", va="bottom", fontsize=8)

fig.legend(loc="upper right")
fig.tight_layout()
plt.show()

print("top-1 share: %.1f%%" % shares[0])
print("top-3 share: %.1f%%" % cum[2])


In [ ]:
# --- Plot 2: zero-purchase ratio ---
info = category_info(rows)
items = sorted(info.items(), key=lambda kv: kv[1]["zero"] / kv[1]["days"], reverse=True)
labels = [short_label(cid, v["code"]) for cid, v in items]
ratios = [100.0 * v["zero"] / v["days"] for _, v in items]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(items))
ax.bar(x, ratios, color="#55A868")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_ylabel("Zero-purchase days share %")
ax.set_title("Zero-purchase ratio (dev period, by category)")
ax.set_ylim(0, 20)
for xi, rv in enumerate(ratios):
    ax.annotate(f"{rv:.1f}%", (xi, rv), ha="center", va="bottom", fontsize=8)
fig.tight_layout()
plt.show()

zero_days = sum(v["zero"] for _, v in info.items())
total_days = sum(v["days"] for _, v in info.items())
print("overall zero-purchase share: %.1f%% (%d/%d category-days)"
      % (100.0 * zero_days / total_days, zero_days, total_days))


In [ ]:
# --- Plot 3: weekday effect ---
per = {i: {"views": 0, "cart": 0, "purchase": 0} for i in range(7)}
days = {i: set() for i in range(7)}
for r in rows:
    w = r["date"].weekday()
    per[w]["views"] += r["views"]
    per[w]["cart"] += r["cart"]
    per[w]["purchase"] += r["purchase"]
    days[w].add(r["date"])

# Mean daily events (not totals), so weekdays with unequal day counts
# (e.g. 5 Tuesdays vs 4 Mondays in a 31-day window) stay comparable.
n_days = np.array([len(days[i]) for i in range(7)], dtype=float)
x = np.arange(7)
purchase = np.array([per[i]["purchase"] for i in range(7)], dtype=float) / n_days
cart = np.array([per[i]["cart"] for i in range(7)], dtype=float) / n_days

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, purchase, marker="o", color="#C44E52", label="Purchase")
ax.plot(x, cart, marker="s", color="#4C72B0", label="Cart")
ax.set_xticks(x)
ax.set_xticklabels(WEEKDAYS)
ax.set_ylabel("Mean daily events (dev period)")
ax.set_title("Weekday effect (dev period)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()


In [ ]:
# --- Plot 4: trend ---
per = defaultdict(lambda: {"views": 0, "cart": 0, "purchase": 0})
for r in rows:
    per[r["date"]]["views"] += r["views"]
    per[r["date"]]["cart"] += r["cart"]
    per[r["date"]]["purchase"] += r["purchase"]

dates = sorted(per)
purchase = np.array([per[d]["purchase"] for d in dates], dtype=float)
x = np.arange(len(dates))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, purchase, marker=".", color="#4C72B0")
ax.set_xticks(x[::5])
ax.set_xticklabels([d.isoformat()[5:] for d in dates[::5]], rotation=45)
ax.set_ylabel("Total purchase events (selected categories)")
ax.set_title("Trend (dev-period daily purchases, selected categories)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

half = len(purchase) // 2
first_mean = purchase[:half].mean()
second_mean = purchase[half:].mean()
print("first-half mean: %.0f" % first_mean)
print("second-half mean: %.0f" % second_mean)
print("change: %+.1f%%" % ((second_mean - first_mean) / first_mean * 100.0))


In [ ]:
# --- Plot 5: view -> cart -> purchase funnel ---
views = sum(r["views"] for r in rows)
cart = sum(r["cart"] for r in rows)
purchase = sum(r["purchase"] for r in rows)
stages = ["View", "Cart", "Purchase"]
values = [views, cart, purchase]

fig, ax = plt.subplots(figsize=(8, 5))
y = np.arange(len(stages))[::-1]  # top = view
ax.barh(y, values, color=["#4C72B0", "#55A868", "#C44E52"], height=0.55)
ax.set_yticks(y)
ax.set_yticklabels(stages)
ax.set_xscale("log")
ax.set_xlabel("Total events (dev period, selected categories, log axis)")
ax.set_title("View -> Cart -> Purchase funnel (dev period)")
stage_ratios = [
    "",
    f" ({cart / views * 100:.1f}% of View)",
    f" ({purchase / cart * 100:.1f}% of Cart / {purchase / views * 100:.1f}% of View)",
]
for yi, v, rt in zip(y, values, stage_ratios):
    ax.annotate(f"{v:,}{rt}", (v, yi), va="center", ha="left", fontsize=9)
fig.tight_layout()
plt.show()

print("cart/view: %.2f%%" % (cart / views * 100))
print("purchase/cart: %.2f%%" % (purchase / cart * 100))
print("purchase/view: %.2f%%" % (purchase / views * 100))


## Conclusions

- Category concentration is extreme: the top-1 category accounts for ~72% of
  dev-period purchases and the top-3 for ~88% of the 10 selected categories —
  a few dominant categories drive most of the target signal.
- Zero-purchase days are rare overall (~1.6% of category-days); one category
  has zero-purchase days.
- Purchases peak mid-week (Wednesday) and trough on Saturday (roughly a 1.3x
  spread), so the weekday feature `cal_target_weekday` is informative.
- Dev-period purchases trend upward overall (~+4% in the second half vs the
  first half).
- The event funnel is very steep: cart is ~4.2% of view, and purchase is ~67%
  of cart (~2.8% of view). These are event-count ratios, NOT unique-user
  conversion rates.

All figures use only the development window (2019-10-01 .. 2019-10-31), so no
validation/test information leaks into this EDA.
